# connexplorer quickstart

Everything below returns polars DataFrames or plain numpy/scipy objects. Root ids are the public identifiers. Build the datasets first (`01_build_datasets.ipynb` or `connexplorer build ...`).

In [ ]:
import connexplorer as cnx
import polars as pl

ds = cnx.open("flywire")   # or a path; reads only the manifest
ds.info()

## Cell types

`ds.types` has one row per type in the order used by the type matrix. `nt_source` tells whether a transmitter comes from the prediction majority or a literature override.

In [ ]:
display(ds.types.head())
ds.types.filter(pl.col("nt_source") == "literature")

## Input profile of a type

`frac_input` is the share of Mi1's total input, `frac_partner_output` the share of the partner type's whole-dataset output that lands on Mi1, `weight_norm` their geometric mean.

In [ ]:
ds["Mi1"].inputs(by="type", normalize=True).head(10)

## A cell-by-cell block

Right-side L1 and Mi1 cells onto right-side L5 cells.

In [ ]:
pre = ds.select(type=["L1", "Mi1"], side="R")
post = ds.select(type="L5", side="R")
blk = ds.connectivity[pre, post]
print(blk, blk.values.shape)
blk.long.head()

## One neuron

A Neuron is a NeuronSet of size one; every query works on both.

In [ ]:
n = ds.select(type="Mi1", side="R")[0]
print(n)
display(n.inputs(by="type").head(10))
n.outputs(min_syn=5).head()

## Synapse locations

Never loaded wholesale: a filter on the sorted column reads one row group.

In [ ]:
syn = n.synapses("in")
print(syn.height, "input synapses")
display(syn.head())
cnx.xyz(syn)[:3]

## Viewer link

URLs are returned, never opened. `partners` adds the top partner types as layers; `synapses` adds their synapse points.

In [ ]:
url = n.view(partners="in", top=5, synapses=True)
print(url[:120], "...")

## Morphology and a passive cable model

Skeletons come from the SWC zip under `data/<dataset>/tables/skeletons/` and are always in micrometres.

In [ ]:
sk = n.skeleton()
comp = cnx.morph.segment(sk, min_length_um=0.5)
comp.summary()
m = cnx.models.Cable(comp, Rm=8000, Ra=400, Cm=0.6)
V = m.steady_state({0: 10e-12})
print(f"V at root {V[0]:.2f} mV, input resistance {m.input_resistance(0):.2f} GOhm")
cnx.viz.plot_voltage(comp, V)

## Two datasets

The Male CNS types table carries the vendor's FlyWire type name, so profiles align on FlyWire names (subtypes are summed).

In [ ]:
mc = cnx.open("mcns")
print(mc.types_like("R7"))
cnx.compare(ds["T4a"], mc["T4a"]).inputs().head(10)